# AWF Fluent LLM Training — Google Colab GPU

**Goal**: Train a fluent AWF LLM that generates real English sentences.

**What you get**: A 1.4M parameter AWF model that generates coherent English text, trained on TinyStories with 10x training speedup.

**Time on Colab T4 GPU**: ~1-2 hours for fluency (50K+ steps)

## Instructions
1. Set Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Training auto-saves every 500 steps
4. You can stop and resume anytime (it picks up from the last checkpoint)
5. You can add your own datasets and continue training

## What makes this fluent (vs the prototype)
- **BPE tokenizer** (1024 vocab) — 4x more efficient than byte-level, learns word patterns
- **Bigger model** (d=384, 8 layers, 1.4M params) — 2.3x more capacity
- **Longer context** (256 tokens) — better sentence completion
- **Output caching** — 10x training speedup so you get more steps per GPU hour
- **Better sampling** — temperature, top-k, top-p, repetition penalty

## Step 1: Setup

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/Deexv/AWF.git
%cd AWF
!pip install -r requirements.txt

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
    print(f'\n✅ GPU ready! Training will be ~10x faster than CPU.')
else:
    print('⚠️ No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## Step 2: Download Dataset

In [ ]:
# Download TinyStories dataset (25MB — real dataset designed for small LMs)
!python scripts/download_tinystories.py

# Verify
with open('data/tinystories_train.txt') as f:
    text = f.read()
print(f'Dataset: {len(text):,} chars, {len(text.split()):,} words')
print(f'Sample: {text[:200]}')

## Step 3: Train the AWF LLM

This trains a 1.4M parameter AWF model with BPE tokenizer and output caching.

**Training plan** (adjust `--epochs` and `--time_budget` as needed):
- Run 1: 30 minutes → ~10K steps (model starts generating words)
- Run 2-3: 30 min each → ~30K steps total (model generates sentences)
- Run 4-5: 30 min each → ~50K+ steps (model generates coherent paragraphs)

**You can stop anytime** — training saves every 500 steps. Just re-run the cell with `--resume` to continue.

In [ ]:
# RUN 1: First training session (30 minutes)
# This will train the BPE tokenizer, build the model, and start training
# With output caching, you get ~10x more steps per minute than standard training

!python scripts/train_fluent.py \
    --epochs 1 \
    --time_budget 1800 \
    --batch_size 32 \
    --lr 1e-3 \
    --save_every 500 \
    --use_bpe \
    --bpe_vocab 1024 \
    --use_output_cache \
    --cache_max_staleness 5 \
    --checkpoint_name awf_fluent.pt

In [ ]:
# RUN 2: Continue training (auto-resumes from checkpoint)
# Just re-run this cell as many times as you want
# Each run gives ~10K more steps

!python scripts/train_fluent.py \
    --resume \
    --epochs 1 \
    --time_budget 1800 \
    --batch_size 32 \
    --lr 5e-4 \
    --save_every 500 \
    --use_output_cache \
    --cache_max_staleness 5 \
    --checkpoint_name awf_fluent.pt

In [ ]:
# RUN 3: Fine-tune with lower learning rate (for final polish)

!python scripts/train_fluent.py \
    --resume \
    --epochs 1 \
    --time_budget 1800 \
    --batch_size 32 \
    --lr 2e-4 \
    --save_every 500 \
    --use_output_cache \
    --cache_max_staleness 5 \
    --checkpoint_name awf_fluent.pt

## Step 4: Test Fluency

After training (at least 30K steps), test if the model generates fluent English.

In [ ]:
# Test fluency — generate text from multiple prompts
!python scripts/chat_fluent.py --prompt "Once upon a time there was a little girl named Lily"
!python scripts/chat_fluent.py --prompt "The sun was shining brightly"
!python scripts/chat_fluent.py --prompt "In a small village"
!python scripts/chat_fluent.py --prompt "A boy named Tom loved to play"
!python scripts/chat_fluent.py --prompt "The old man smiled and said"

# If the text is not fluent yet, run more training cells above (Step 3)

## Step 5: Interactive Chat

Chat with your trained model interactively.

In [ ]:
# Interactive chat (type prompts and get responses)
# Note: In Colab, use input() carefully. For batch generation use the cell above.

import sys, os, json, torch, torch.nn.functional as F
sys.path.insert(0, '.')
from awf.core import AWFTransformer, num_params
from scripts.train_fluent import BPETokenizer, ByteTokenizer, generate, BLOCK

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('checkpoints/awf_fluent.pt', map_location=device)
cfg = ckpt['config']

model = AWFTransformer(
    vocab_size=cfg['vocab_size'], d_model=cfg['d_model'],
    n_layers=cfg['n_layers'], n_heads=cfg['n_heads'],
    block_size=cfg['block_size'], residual_rank=cfg['residual_rank'],
    sparse_k=cfg['sparse_k'],
    gen_kwargs=dict(n_fourier=16, hidden=128, n_layers=3), gen_grid=16
).to(device)
model.load_state_dict(ckpt['model'])
model.eval()

# Load tokenizer
if ckpt.get('tokenizer'):
    tokenizer = BPETokenizer()
    tokenizer.from_dict(ckpt['tokenizer'])
else:
    tokenizer = ByteTokenizer()

print(f'AWF model loaded: {num_params(model):,} params, step {ckpt["step"]}')
print(f'Val loss: {ckpt.get("val_loss", "?")}')
print(f'\nType a prompt and press Enter (type "quit" to exit):\n')

while True:
    prompt = input('You> ').strip()
    if prompt.lower() in ('quit', 'exit', 'q'):
        break
    if not prompt:
        continue
    response = generate(model, tokenizer, device, prompt, n_tokens=200,
                        temperature=0.7, top_k=30, top_p=0.9, repetition_penalty=1.2)
    print(f'AI> {response}\n')

## Step 6: Train on YOUR OWN Dataset (Optional)

You can continue training on a different dataset. The model will learn from the new data while keeping what it already learned.

Upload your text file to Colab, then run:

In [ ]:
# Upload your own text file
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    print(f'Uploaded: {filename} ({len(uploaded[filename])} bytes)')
    # Move to data directory
    import shutil
    shutil.move(filename, f'data/{filename}')
    print(f'Moved to data/{filename}')

In [ ]:
# Continue training on TinyStories + YOUR dataset
# Replace 'your_file.txt' with your uploaded filename

!python scripts/train_fluent.py \
    --resume \
    --datasets data/tinystories_train.txt data/your_file.txt \
    --epochs 1 \
    --time_budget 1800 \
    --batch_size 32 \
    --lr 3e-4 \
    --save_every 500 \
    --use_output_cache \
    --cache_max_staleness 5 \
    --checkpoint_name awf_fluent.pt

## Step 7: Save to Google Drive (Optional)

Save your trained model so you don't lose it when Colab disconnects.

In [ ]:
# Mount Google Drive and save checkpoint
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/AWF
!cp checkpoints/awf_fluent.pt /content/drive/MyDrive/AWF/
!cp checkpoints/bpe_fluent.json /content/drive/MyDrive/AWF/
!cp benchmarks/fluent_training_log.json /content/drive/MyDrive/AWF/
print('✅ Saved to Google Drive: /content/drive/MyDrive/AWF/')
print('You can restore from here in a new Colab session:')
print('  !cp /content/drive/MyDrive/AWF/awf_fluent.pt checkpoints/')
print('  !cp /content/drive/MyDrive/AWF/bpe_fluent.json checkpoints/')
print('  !python scripts/train_fluent.py --resume --epochs 1 --time_budget 1800')

## Step 8: Restore from Google Drive (New Session)

If Colab disconnected and you want to continue training in a new session:

In [ ]:
# In a NEW Colab session, run Step 1 first, then:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/AWF/awf_fluent.pt checkpoints/
!cp /content/drive/MyDrive/AWF/bpe_fluent.json checkpoints/
print('✅ Checkpoint restored from Google Drive')

# Now continue training:
!python scripts/train_fluent.py \
    --resume \
    --epochs 1 \
    --time_budget 1800 \
    --batch_size 32 \
    --lr 2e-4 \
    --use_output_cache \
    --cache_max_staleness 5

## Tips for Better Fluency

1. **Train longer**: 50K+ steps is the minimum for coherent sentences. 100K+ for paragraphs.
2. **Lower learning rate over time**: Start with 1e-3, then 5e-4, then 2e-4, then 1e-4
3. **More data**: Upload your own text (books, articles, dialogue) and continue training
4. **Use the output caching**: It gives 10x more steps per GPU hour. Keep `--cache_max_staleness 5`
5. **Check the samples**: The script generates samples every 500 steps. Watch them improve.
6. **Don't use extreme caching** (ms=9999) for fluency — it freezes the transformer too early. Use ms=5 (conservative).

## How to Know When It's Fluent

Run `scripts/chat_fluent.py --prompt "Once upon a time"` after each training session.
When the output contains:
- Complete sentences (subject + verb + object)
- Correct grammar (most of the time)
- Coherent story structure (beginning, middle, end)
- Named characters with consistent behavior

...then it's fluent. This typically happens around 50K-100K steps with this architecture.